In [ ]:
# ==============================================================================
# PIPELINE DE EXTRAÇÃO, HIGIENIZAÇÃO E SALVAMENTO DA BASE LIMPA (NHIS 2025)
# ==============================================================================
import numpy as np
import pandas as pd

YES_NO = {
    1 : "SIM",
    2 : "NÃO",
    7 : np.nan,
    8 : np.nan,
    9 : np.nan
}

QTD = {
    1: "1",
    2: "2",
    3: "3+",
    4: "INCERTO"
}

# 1. Dicionário Consolidado das Variáveis Selecionadas (101 Features + Controles)
MAPA_VARIAVEIS_PAPER = {
    # Controles amostrais
    "HHX": "id_domicilio", 
    "WTFA_A": "peso_amostral",
    #dados pessoais
    "SEX_A": "sexo",
    "URBRRL23": "classificacao_urbano_rural",
    "REGION": "regiao_geografica",
    "AGE65": "faixa_65_mais",
    #dados família
    "MARITAL_A": "estado_civil_declarado",
    "MARSTAT_A": "estado_civil",
    "PARSTAT_A": "status_parental", 
    "SPOUSESEX_A": "sexo_conjuge",
    "SPOUSEP_A": "separacao_legal_conjugal",
    "SPOUSWRK_A": "cojuge_trabalha",
    "OVER65FLG_A": "flg_idoso_familia",
    "PCNTADLT_A": "qtd_adultos_familia",
    "PCNTKIDS_A": "qtd_crianças_familia",
    "MAXEDUCP_A": "escolaridade_maxima_familia",
    
    #moradia
    "HOUYRSLIV_A": "tempo_moradia_anos",
    #"HOUSECOST_A": "dificuldade_custo_moradia", #não estava no paper e não tem em 2025
    "HOUTENURE_A": "tipo_posse_imovel", 
    "YRSINUS_A": "qtd_anos_estados_unidos",


#######################################################################################


    #economicos
    "FLUNCH12M1_A":"merenda_escolar_gratuita",
    "RATCAT_A": "categoria_razao_pobreza",
    "POVRATTC_A": "razao_renda_pobreza",
    "INCINTER_A":"renda_investimentos",
    "HISTOPJOB_A":"perdeu_plano_demitido_ou_trocou_emprego",
    "PAYWORRY_A": "estresse_financeiro_medico",
    "PAYNOBLLNW_A": "dividas_medicas_em_aberto",
    "RSNHICOST_A": "sem_plano_por_custo_inacessivel",

    #plano de saude
    "OPDEDUC_A": "plano_estadual_dedutivel",
    "PRDEDUC1_A": "plano_privado_dedutivel",
    "PRDEDUC2_A": "plano_privado_dedutivel2",
    "VAHOSP_A": "plano_militar_saude",
    "PLN1PAY2_A": "plano_patrao_sindicato_pagam",
    "HINOTMYR_A": "plano_meses_sem", #


#######################################################################


    #saude geral
    "PHSTAT_A": "autoavaliacao_saude_geral", 
    "BMICAT_A": "categoria_imc",
    "WEIGHTLBTC_A": "peso_libras",
    "LSATIS4_A": "satisfacao_com_a_vida",
    "SOCWRKLIM_A": "limitacao_trabalho_por_saude",
    
    "ASTILL_A": "asma_ativa_atualmente",
    "HYPEV_A": "historico_hipertensao",
    "CHLEV_A": "historico_colesterol",
    "CHLMED_A": "remedio_colesterol",
    "DIBINSSTYR_A": "interrompeu_insulina_prm_ano",
    "ARTHEV_A": "historico_artrite",
    "DIBEV_A": "historico_diabetes",
    "DIBPILL_A": "med_diabetes_oral", # no paper usou dibpill mas tem vários outros tratamentos

    "PAIAPG3M_A": "dor_abdominal_cronica",

    
    "TBIHLSBMC_A": "concussao_sintomas_pos_trauma", #não tem no & artigo não tem em 2025
    #"INJFALL_A": "lesao_decorrente_queda",#não tem no artigo & não tem em 2025
    #"INJFALLHOM_A": "queda_ocorrida_em_casa", #não tem no artigo & não tem em 2025

    #"REPWRKCAUS_A": "ler_dort_causada_no_trabalho", não tem em 2025
    #"REPSTRAIN_A": "lesao_por_esforco_repetitivo", não tem em 2025
    #"REPLIMIT_A": "limitacao_por_ler_dort", não tem em 2025

    #câncer
    "CERVIAGETC_A": "idade_cancer_colo_utero",
    "PANCRAGETC_A": "idade_cancer_pancreas", #
    "ESOPHAGETC_A": "idade_cancer_esofago", #
    "LUNGAGETC_A": "idade_cancer_pulmao",
    "LIVERAGETC_A": "idade_cancer_figado",
    "SKNDKAGETC_A": "idade_cancer_pele_indeterminado",
    "THYROAGETC_A": "idade_cancer_tireoide",
    "STOMAAGETC_A": "idade_cancer_estomago",


    #vacinação

    "SHTFLUY_A": "ano_vacina_gripe",
    "SHTFLU12M_A": "vacina_gripe_ultimos_12m",

    "SHTPNEUNB_A": "qtd_vacina_pneumonia",
    "SHTTETANUS_A": "vacina_tetanus_10a",#


    #"INJWRKDYTC_A": "problema_saude_dias_sem_trabalhar"



###################################################################################################
    #Uso de recursos de saude ####################################################################

    "URGCC12MTC_A": "visitas_pronto_atendimento_12m",
    "EMERG12MTC_A": "visitas_emergencia_hospitalar_12m",
    "EYEEX12M_A": "visitas_exame_vista_12m",
    "WELLVIS_A": "tempo_ultimo_checkup_geral",
    "WELLNESS_A": "ultima_visita_foi_checkup", #meio redundante

    #"INJSAWDOC_A": "atendimento_medico_por_lesao", não tem e, 2025


##################################################################################################




    #Ansiedade, Depressão e seus Sintomas ########################################################

    "DEPEV_A": "historico_depressao",
    "DEPMED_A": "remedio_depressão", #pode enviesar muito!
    "DEPFREQ_A": "freq_depressao",
    "DEPLEVEL_A": "intensidade_depressao",
    "PHQ81_A": "freq_perda_interesse_2sem",
    "PHQ82_A": "freq_sentiu_deprimido_2sem",
    #"HOPELESS_A": "k6_freq_desesperanca", #infelizmente não tem 2025. PHQ82_A equivalente?
    #"WORTHLESS_A": "k6_freq_desvalorizacao", #infelizmente não tem 2025. PHQ82_A equivalente?
    "GAD72_A": "freq_preocupacao_incontrolavel_2sem",
    "PHQ83_A": "freq_dificuldade_dormir_2sem",
    #"SLPFLL_A": "dificuldade_adormecer", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPSTY_A": "dificuldade_manter_sono", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPREST_A": "acorda_descansado", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    "PHQ84_A": "freq_cansaso_2sem",
    #"FGEFRQTRD_A": "frequencia_fadiga_3m", #infelizmente não tem no paper e 2025. PHQ84_A equivalente?
    "PHQ85_A": "freq_hiporexia_compulsao_alimentar_2sem",
    "PHQ86_A": "freq_autoavaliacao_negtiva_2sem", #auto estima, etc...
    "PHQ87_A": "freq_dificuldade_concentracao_2sem",
    #"SAD_A": "k6_freq_tristeza_profunda", #infelizmente não tem 2025. PHQ82_A equivalente?
    "LONELY_A": "freq_solitário", #não está no paper mas deveria
    #"EFFORT_A": "k6_freq_esforco_extremo", não está no paper. Não achei equivalente
    #"PHQCAT_A": "escala_sintomas_depressão", #pode enviesar muito!
    
    "COGFRQDFF_A": "frequencia_falha_memoria", 
    "COGTYPEDFF_A": "tipo_dificuldade_cognitiva",
    "COGMEMDFF_A": "qtd_dificuldade_cognitiva",

    "ANXEV_A": "historico_transtorno_ansiedade",
    #"ANXMED_A": "med_ansiedade", #pode enviesar muito!
    "ANXFREQ_A": "freq_ansiedade",
    "ANXLEVEL_A": "intensidade_ansiedade",

    "GAD71_A": "freq_ansiedade_2sem",
    #"NERVOUS_A": "k6_freq_nervosismo",
    "GAD72_A": "freq_preocupacao_descontrolada_2sem",
    "GAD73_A": "freq_preocupacao_desnecessaria_2sem",
    "GAD74_A": "freq_dificuldade_descansar_2sem",
    "GAD75_A": "freq_agitacao_inquieta", # não tinha no paper mas é de ansiedade

    #"GADCAT_A": "escala_sintomas_ansiedade", #pode enviesar muito!
    #"K6SPD_A": "sofrimento_psicologico_k6", #não tem no paper nem 2025. Pode enviesar?
    ##############################################################################################################
    
    # Segurança alimentar: INFELIZMENTE NÃO TEM EM 2025
    #"FDSLESS_A": "comeu_menos_que_devia", 
    #"FDSWEIGHT_A": "perdeu_peso_restricao_alimentar", 
    #"FDSSKIP_A": "diminuiu_ou_pulou_refeicoes",
    #"FDSHUNGRY_A": "passou_fome_por_falta_dinheiro",
    
    #################################################################################################################
    
    #Dificuldade de andar
    "NOEQWLK13M_A": "andar_dificuldade_500m_sem_aparelho",
    "NOEQSTEPS_A": "andar_dificuldade_degraus_sem_aparelho",
    "EQWLK13M_A": "andar_dificuldade_500m_com_aparelho",
    "EQSTEPS_A": "andar_dificuldade_degraus_com_aparelho",
    "PERASST_A": "andar_necessita_ajuda_outra_pessoa",
    
    #####################################################################
    
    #
    
    "SUPPORT_A": "suporte_social", #não tem no artigo mas deveria

    #SCONNECT1_A    Number of times you get together with people you are close to 
    #SCONNECT2_A    Number of times you talk to people on telephone or by video 
    #SCONNECT3_A    Number of times you attend religious services 
    #SCONNECT4_A    Number of times you attend meetings of clubs or organizations 
    #PARENTCON1_A   How often receive social/emotional support parenting 
    #PARENTCON2_A   How difficult are day-to-day demands parenting 
    #PARENTCON3_A   How often do you talk, chat, or connect with other parents 
    
    #################################################################################################
    #Uso de cigarro

    "SMKNOW_A": "fuma_cigarro_atualmente",
    "CIGNOW_A": "qtd_cigarros_dia",
    "SMK30D_A": "dias_fumados_mes",
    "CIG30D_A": "qtd_cigarro_mes",
    "ECIGNOW_A": "usa_cigarro_eletronico_atualmente",
    "SMKEV_A": "ja_fumante", # na verdade foi perguntado já fumou 100 cigarros na vida. Isso categoriza como fumante? 

    
    
}

colunas_para_ler = list(MAPA_VARIAVEIS_PAPER.keys())
print(f"Colunas a serem carregadas: {len(colunas_para_ler)}")

# 2. Leitura otimizada direto do CSV
df_bruto = pd.read_csv("dados/adult25.csv", usecols=colunas_para_ler, low_memory=False)
print(f"Base carregada: {df_bruto.shape[0]:,} linhas e {df_bruto.shape[1]} colunas.")

# 3. Construção do Desfecho Clínico (target_medicacao)
# 1 = Sim, 2 = Não, outros = Indeterminado
cond_sim = (df_bruto["DEPMED_A"] == 1) | (df_bruto["ANXMED_A"] == 1)
cond_nao = (df_bruto["DEPMED_A"] == 2) & (df_bruto["ANXMED_A"] == 2)
df_bruto["target_medicacao"] = np.where(cond_sim, 1, np.where(cond_nao, 0, np.nan))

# 4. Aplicação dos Nomes Padronizados em Português
df_limpo = df_bruto.rename(columns=MAPA_VARIAVEIS_PAPER)

# 5. Exportação para CSV limpo
nome_arquivo_saida = "dado_ingerido/adult25_limpo_selecionadas.csv"
df_limpo.to_csv(nome_arquivo_saida, index=False)

print(f"\nBase limpa exportada como '{nome_arquivo_saida}' com sucesso!")
print(f"Dimensões finais: {df_limpo.shape[0]:,} participantes e {df_limpo.shape[1]} atributos.")
print("\nDistribuição da variável dependente (target_medicacao):")
print(df_limpo["target_medicacao"].value_counts(dropna=False, normalize=True).round(4) * 100)

Colunas a serem carregadas: 101
Base carregada: 24,215 linhas e 101 colunas.


C:\Users\theus\AppData\Local\Temp\ipykernel_4164\3879145305.py:243: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_bruto["target_medicacao"] = np.where(cond_sim, 1, np.where(cond_nao, 0, np.nan))



Base limpa exportada como 'dado_ingerido/adult25_limpo_selecionadas.csv' com sucesso!
Dimensões finais: 24,215 participantes e 102 atributos.

Distribuição da variável dependente (target_medicacao):
target_medicacao
0.0    81.09
1.0    17.00
NaN     1.91
Name: proportion, dtype: float64


In [29]:
df_limpo

,categoria_razao_pobreza,qtd_anos_estados_unidos,estado_civil,cojuge_trabalha,sexo_conjuge,categoria_imc,peso_libras,visitas_pronto_atendimento_12m,visitas_emergencia_hospitalar_12m,escolaridade_maxima_familia,...,asma_ativa_atualmente,remedio_colesterol,historico_colesterol,historico_hipertensao,satisfacao_com_a_vida,autoavaliacao_saude_geral,peso_amostral,id_domicilio,razao_renda_pobreza,target_medicacao
0,6,NaN,1,2.0,1.0,3,171,0,0,4,...,NaN,NaN,2,2,2,3,10636.862,H012128,1.58,1.0
1,12,NaN,5,NaN,NaN,2,140,0,0,3,...,NaN,2.0,1,1,1,4,2996.860,H000617,4.32,0.0
2,1,NaN,7,NaN,NaN,3,175,0,0,2,...,NaN,NaN,2,2,2,3,29032.445,H033243,0.18,0.0
3,14,NaN,1,2.0,2.0,3,215,0,0,5,...,NaN,1.0,1,1,1,4,7875.107,H049140,6.32,0.0
4,13,NaN,7,NaN,NaN,4,198,0,0,8,...,NaN,1.0,1,1,2,2,4824.620,H012797,4.90,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24210,14,NaN,1,1.0,1.0,4,200,0,0,9,...,NaN,NaN,2,1,1,3,4869.749,H014073,6.18,0.0
24211,4,NaN,5,NaN,NaN,4,285,0,2,5,...,NaN,NaN,2,2,2,2,3350.243,H029805,1.21,0.0
24212,12,NaN,4,NaN,NaN,4,210,0,2,4,...,NaN,NaN,2,1,1,5,2374.594,H003491,4.32,0.0
24213,9,NaN,1,2.0,1.0,3,165,0,0,5,...,NaN,1.0,1,2,1,2,2400.569,H001251,2.63,0.0


Essas são as variáveis de características demográficas, gênero e suporte familiar

In [ ]:
ID_DOMICILIO = "ID_DOMICILIO"
PESO_AMOSTRAL = "PESO_AMOSTRAL"
SEXO = "SEXO"
CLASSIFICACAO_URBANO_RURAL = "CLASSIFICACAO_URBANO_RURAL"
REGIAO_GEOGRAFICA = "REGIAO_GEOGRAFICA"
FAIXA_65_MAIS = "FAIXA_65_MAIS"
ESTADO_CIVIL_DECLARADO= "ESTADO_CIVIL_DECLARADO"
ESTADO_CIVIL= "ESTADO_CIVIL"
STATUS_PARENTAL= "STATUS_PARENTAL"
SEXO_CONJUGE= "SEXO_CONJUGE"
SEPARACAO_LEGAL_CONJUGAL= "SEPARACAO_LEGAL_CONJUGAL"
COJUGE_TRABALHA= "COJUGE_TRABALHA"
FLG_IDOSO_FAMILIA= "FLG_IDOSO_FAMILIA"
QTD_ADULTOS_FAMILIA= "QTD_ADULTOS_FAMILIA"
QTD_CRIANÇAS_FAMILIA= "QTD_CRIANÇAS_FAMILIA"
ESCOLARIDADE_MAXIMA_FAMILIA= "ESCOLARIDADE_MAXIMA_FAMILIA"
TEMPO_MORADIA_ANOS= "TEMPO_MORADIA_ANOS"
TIPO_POSSE_IMOVEL= "TIPO_POSSE_IMOVEL"
QTD_ANOS_ESTADOS_UNIDOS= "QTD_ANOS_ESTADOS_UNIDOS"
SUPORTE_SOCIAL= "SUPORTE_SOCIAL"

mapa_limpo = {
  # Controles amostrais
    "HHX": ID_DOMICILIO,
    "WTFA_A": PESO_AMOSTRAL,
    #dados pessoais
    "SEX_A": SEXO,
    "URBRRL23": CLASSIFICACAO_URBANO_RURAL,
    "REGION": REGIAO_GEOGRAFICA,
    "AGE65": FAIXA_65_MAIS,
    #dados família
    "MARITAL_A": ESTADO_CIVIL_DECLARADO,
    "MARSTAT_A": ESTADO_CIVIL,
    "PARSTAT_A": STATUS_PARENTAL, 
    "SPOUSESEX_A": SEXO_CONJUGE,
    "SPOUSEP_A": SEPARACAO_LEGAL_CONJUGAL,
    "SPOUSWRK_A": COJUGE_TRABALHA,
    "OVER65FLG_A": FLG_IDOSO_FAMILIA,
    "PCNTADLT_A": QTD_ADULTOS_FAMILIA,
    "PCNTKIDS_A": QTD_CRIANÇAS_FAMILIA,
    "MAXEDUCP_A": ESCOLARIDADE_MAXIMA_FAMILIA,
    
    #moradia
    "HOUYRSLIV_A": TEMPO_MORADIA_ANOS,
    #"HOUSECOST_A": "dificuldade_custo_moradia", #não estava no paper e não tem em 2025
    "HOUTENURE_A": TIPO_POSSE_IMOVEL, 
    "YRSINUS_A": QTD_ANOS_ESTADOS_UNIDOS,

    "SUPPORT_A": SUPORTE_SOCIAL, #não tem no artigo mas deveria
}

Essas são as variavais de saúde geral

In [ ]:
AUTOAVALIACAO_SAUDE_GERAL = "AUTOAVALIACAO_SAUDE_GERAL"
CATEGORIA_IMC = "CATEGORIA_IMC"
PESO_LIBRAS = "PESO_LIBRAS"
SATISFACAO_COM_A_VIDA = "SATISFACAO_COM_A_VIDA"
LIMITACAO_TRABALHO_POR_SAUDE ="LIMITACAO_TRABALHO_POR_SAUDE"
ASMA_ATIVA_ATUALMENTE ="ASMA_ATIVA_ATUALMENTE"
HISTORICO_HIPERTENSAO ="HISTORICO_HIPERTENSAO"
HISTORICO_COLESTEROL ="HISTORICO_COLESTEROL"
REMEDIO_COLESTEROL= "REMEDIO_COLESTEROL"
INTERROMPEU_INSULINA_PRM_ANO = "INTERROMPEU_INSULINA_PRM_ANO"
HISTORICO_ARTRITE = "HISTORICO_ARTRITE"
HISTORICO_DIABETES = "HISTORICO_DIABETES"
MED_DIABETES_ORAL = "MED_DIABETES_ORAL"
DOR_ABDOMINAL_CRONICA = "DOR_ABDOMINAL_CRONICA"
CONCUSSAO_SINTOMAS_POS_TRAUMA ="CONCUSSAO_SINTOMAS_POS_TRAUMA"
IDADE_CANCER_COLO_UTERO = "IDADE_CANCER_COLO_UTERO"
IDADE_CANCER_PANCREAS ="IDADE_CANCER_PANCREAS"
IDADE_CANCER_ESOFAGO = "IDADE_CANCER_ESOFAGO"
IDADE_CANCER_PULMAO = "IDADE_CANCER_PULMAO"
IDADE_CANCER_FIGADO = "IDADE_CANCER_FIGADO"
IDADE_CANCER_PELE_INDETERMINADO = "IDADE_CANCER_PELE_INDETERMINADO"
IDADE_CANCER_TIREOIDE = "IDADE_CANCER_TIREOIDE"
IDADE_CANCER_ESTOMAGO = "IDADE_CANCER_ESTOMAGO"

ANO_VACINA_GRIPE = "ANO_VACINA_GRIPE"
VACINA_GRIPE_ULTIMOS_12M = "VACINA_GRIPE_ULTIMOS_12M"
QTD_VACINA_PNEUMONIA = "QTD_VACINA_PNEUMONIA"
VACINA_TETANUS_10A = "VACINA_TETANUS_10A"
ANDAR_DIFICULDADE_500M_SEM_APARELHO = "ANDAR_DIFICULDADE_500M_SEM_APARELHO"
ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO = "ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO"
ANDAR_DIFICULDADE_500M_COM_APARELHO = "ANDAR_DIFICULDADE_500M_COM_APARELHO"
ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO = "ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO"
ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA = "ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA"



mapa_limpo.update(
   {
        #saude geral
        "PHSTAT_A": AUTOAVALIACAO_SAUDE_GERAL, 
        "BMICAT_A": CATEGORIA_IMC,
        "WEIGHTLBTC_A": PESO_LIBRAS,
        "LSATIS4_A": SATISFACAO_COM_A_VIDA,
        "SOCWRKLIM_A": LIMITACAO_TRABALHO_POR_SAUDE,
        
        "ASTILL_A": ASMA_ATIVA_ATUALMENTE,
        "HYPEV_A": HISTORICO_HIPERTENSAO,
        "CHLEV_A": HISTORICO_COLESTEROL,
        "CHLMED_A": REMEDIO_COLESTEROL,
        "DIBINSSTYR_A": INTERROMPEU_INSULINA_PRM_ANO,
        "ARTHEV_A": HISTORICO_ARTRITE,
        "DIBEV_A": HISTORICO_DIABETES,
        "DIBPILL_A": MED_DIABETES_ORAL, # no paper usou dibpill mas tem vários outros tratamentos

        "PAIAPG3M_A": DOR_ABDOMINAL_CRONICA,

        
        "TBIHLSBMC_A": CONCUSSAO_SINTOMAS_POS_TRAUMA, #não tem no & artigo não tem em 2025
        #"INJFALL_A": "lesao_decorrente_queda",#não tem no artigo & não tem em 2025
        #"INJFALLHOM_A": "queda_ocorrida_em_casa", #não tem no artigo & não tem em 2025

        #"REPWRKCAUS_A": "ler_dort_causada_no_trabalho", não tem em 2025
        #"REPSTRAIN_A": "lesao_por_esforco_repetitivo", não tem em 2025
        #"REPLIMIT_A": "limitacao_por_ler_dort", não tem em 2025

        #vacinação

        "SHTFLUY_A": ANO_VACINA_GRIPE,
        "SHTFLU12M_A": VACINA_GRIPE_ULTIMOS_12M,

        "SHTPNEUNB_A": QTD_VACINA_PNEUMONIA,
        "SHTTETANUS_A": VACINA_TETANUS_10A,#


        #"INJWRKDYTC_A": "problema_saude_dias_sem_trabalhar"

        #Difivuldade de andar
        "NOEQWLK13M_A": ANDAR_DIFICULDADE_500M_SEM_APARELHO,
        "NOEQSTEPS_A": ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO,
        "EQWLK13M_A": ANDAR_DIFICULDADE_500M_COM_APARELHO,
        "EQSTEPS_A": ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO,
        "PERASST_A": ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA,

   } 
)


As variáveis: CERVIAGETC_A, PANCRAGETC_A, ESOPHAGETC_A, LUNGAGETC_A, LIVERAGETC_A, SKNDKAGETC_A, SKNDKAGETC_A, THYROAGETC_A, STOMAAGETC_A,    foram retiradas devido as sua grande quantidade de valores nulos, sendo impossível a sua utilização em treinamento pois não pode ser preenchida com valores Moda.

Dos 24.215 mil registros as variáveis apresentaram-se apenas:

CERVIAGETC_A : 13
PANCRAGETC_A : 13
ESOPHAGETC_A : 17
LUNGAGETC_A  : 120
LIVERAGETC_A : 22
SKNDKAGETC_A : 114
THYROAGETC_A : 108
STOMAAGETC_A : 20

Essas variáveis foram substituidas pela variável CANEV_A : "já teve cancer?"
CANEV_A foi preenchida em todos os registros, com a seguinte frequencia de respostas abaixo

Yes 3378  13.95%
No  20804 85.91%


In [ ]:
#IDADE_CANCER_COLO_UTERO = "IDADE_CANCER_COLO_UTERO"
#IDADE_CANCER_PANCREAS ="IDADE_CANCER_PANCREAS"
#IDADE_CANCER_ESOFAGO = "IDADE_CANCER_ESOFAGO"
#IDADE_CANCER_PULMAO = "IDADE_CANCER_PULMAO"
#IDADE_CANCER_FIGADO = "IDADE_CANCER_FIGADO"
#IDADE_CANCER_PELE_INDETERMINADO = "IDADE_CANCER_PELE_INDETERMINADO"
#IDADE_CANCER_TIREOIDE = "IDADE_CANCER_TIREOIDE"
#IDADE_CANCER_ESTOMAGO = "IDADE_CANCER_ESTOMAGO"

TEVE_CANCER = "TEVE_CANCER"


mapa_limpo.update(
   {
        #câncer

        #"CERVIAGETC_A": IDADE_CANCER_COLO_UTERO,
        #"PANCRAGETC_A": IDADE_CANCER_PANCREAS, #
        #"ESOPHAGETC_A": IDADE_CANCER_ESOFAGO, #
        #"LUNGAGETC_A": IDADE_CANCER_PULMAO,
        #"LIVERAGETC_A": IDADE_CANCER_FIGADO,
        #"SKNDKAGETC_A": IDADE_CANCER_PELE_INDETERMINADO,
        #"THYROAGETC_A": IDADE_CANCER_TIREOIDE,
        #"STOMAAGETC_A": IDADE_CANCER_ESTOMAGO,

   } 
)

Essas são as variavais de condições econômicas e plano de saúde

In [ ]:
MERENDA_ESCOLAR_GRATUITA = "MERENDA_ESCOLAR_GRATUITA"
CATEGORIA_RAZAO_POBREZA= "CATEGORIA_RAZAO_POBREZA"
RAZAO_RENDA_POBREZA= "RAZAO_RENDA_POBREZA"
RENDA_INVESTIMENTOS="RENDA_INVESTIMENTOS"
PERDEU_PLANO_DEMITIDO_OU_TROCOU_EMPREGO="PERDEU_PLANO_DEMITIDO_OU_TROCOU_EMPREGO"
ESTRESSE_FINANCEIRO_MEDICO="ESTRESSE_FINANCEIRO_MEDICO"
DIVIDAS_MEDICAS_EM_ABERTO="DIVIDAS_MEDICAS_EM_ABERTO"
SEM_PLANO_POR_CUSTO_INACESSIVEL="SEM_PLANO_POR_CUSTO_INACESSIVEL"

PLANO_ESTADUAL_DEDUTIVEL="PLANO_ESTADUAL_DEDUTIVEL"
PLANO_PRIVADO_DEDUTIVEL="PLANO_PRIVADO_DEDUTIVEL"
PLANO_PRIVADO_DEDUTIVEL2="PLANO_PRIVADO_DEDUTIVEL2"
PLANO_MILITAR_SAUDE="PLANO_MILITAR_SAUDE"
PLANO_PATRAO_SINDICATO_PAGAM="PLANO_PATRAO_SINDICATO_PAGAM"
PLANO_MESES_SEM="PLANO_MESES_SEM"
VISITAS_PRONTO_ATENDIMENTO_12M="VISITAS_PRONTO_ATENDIMENTO_12M"

VISITAS_EMERGENCIA_HOSPITALAR_12M="VISITAS_EMERGENCIA_HOSPITALAR_12M"
VISITAS_EXAME_VISTA_12M="VISITAS_EXAME_VISTA_12M"
TEMPO_ULTIMO_CHECKUP_GERAL="TEMPO_ULTIMO_CHECKUP_GERAL"
ULTIMA_VISITA_FOI_CHECKUP="ULTIMA_VISITA_FOI_CHECKUP"



mapa_limpo.update(
    {
        #economicos
        "FLUNCH12M1_A": MERENDA_ESCOLAR_GRATUITA,
        "RATCAT_A": CATEGORIA_RAZAO_POBREZA,
        "POVRATTC_A": RAZAO_RENDA_POBREZA,
        "INCINTER_A":RENDA_INVESTIMENTOS,
        "HISTOPJOB_A":PERDEU_PLANO_DEMITIDO_OU_TROCOU_EMPREGO,
        "PAYWORRY_A": ESTRESSE_FINANCEIRO_MEDICO,
        "PAYNOBLLNW_A": DIVIDAS_MEDICAS_EM_ABERTO,
        "RSNHICOST_A": SEM_PLANO_POR_CUSTO_INACESSIVEL,

        #plano de saude
        "OPDEDUC_A": PLANO_ESTADUAL_DEDUTIVEL,
        "PRDEDUC1_A": PLANO_PRIVADO_DEDUTIVEL,
        "PRDEDUC2_A": PLANO_PRIVADO_DEDUTIVEL2,
        "VAHOSP_A": PLANO_MILITAR_SAUDE,
        "PLN1PAY2_A": PLANO_PATRAO_SINDICATO_PAGAM,
        "HINOTMYR_A": PLANO_MESES_SEM, #
        "URGCC12MTC_A": VISITAS_PRONTO_ATENDIMENTO_12M,
        #Uso de recursos de saude ####################################################################
        "EMERG12MTC_A": VISITAS_EMERGENCIA_HOSPITALAR_12M,
        "EYEEX12M_A": VISITAS_EXAME_VISTA_12M,
        "WELLVIS_A": TEMPO_ULTIMO_CHECKUP_GERAL,
        "WELLNESS_A": ULTIMA_VISITA_FOI_CHECKUP, #meio redundante

        #"INJSAWDOC_A": "atendimento_medico_por_lesao", não tem e, 2025
    }
)


Essas são condições de estilo de vida e padrões comportamentais

In [ ]:
FUMA_CIGARRO_ATUALMENTE= "FUMA_CIGARRO_ATUALMENTE"
QTD_CIGARROS_DIA= "QTD_CIGARROS_DIA"
DIAS_FUMADOS_MES= "DIAS_FUMADOS_MES"
QTD_CIGARRO_MES= "QTD_CIGARRO_MES"
USA_CIGARRO_ELETRONICO_ATUALMENTE= "USA_CIGARRO_ELETRONICO_ATUALMENTE"
JA_FUMANTE= "JA_FUMANTE"

mapa_limpo.update(
    {
        "SMKNOW_A": FUMA_CIGARRO_ATUALMENTE,
        "CIGNOW_A": QTD_CIGARROS_DIA,
        "SMK30D_A": DIAS_FUMADOS_MES,
        "CIG30D_A": QTD_CIGARRO_MES,
        "ECIGNOW_A": USA_CIGARRO_ELETRONICO_ATUALMENTE,
        "SMKEV_A": JA_FUMANTE, # na verdade foi perguntado já fumou 100 cigarros na vida. Isso categoriza como fumante? 
    }
)


Essas são condições relacionadas a ansiedade e depressão

In [1]:
HISTORICO_DEPRESSAO= "HISTORICO_DEPRESSAO"
REMEDIO_DEPRESSÃO= "REMEDIO_DEPRESSÃO"
FREQ_DEPRESSAO= "FREQ_DEPRESSAO"
INTENSIDADE_DEPRESSAO= "INTENSIDADE_DEPRESSAO"
FREQ_PERDA_INTERESSE_2SEM= "FREQ_PERDA_INTERESSE_2SEM"
FREQ_SENTIU_DEPRIMIDO_2SEM= "FREQ_SENTIU_DEPRIMIDO_2SEM"


FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM= "FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM"
FREQ_DIFICULDADE_DORMIR_2SEM= "FREQ_DIFICULDADE_DORMIR_2SEM"
FREQ_CANSASO_2SEM= "FREQ_CANSASO_2SEM"
FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM= "FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM"
FREQ_AUTOAVALIACAO_NEGTIVA_2SEM= "FREQ_AUTOAVALIACAO_NEGTIVA_2SEM"
FREQ_DIFICULDADE_CONCENTRACAO_2SEM= "FREQ_DIFICULDADE_CONCENTRACAO_2SEM"


FREQ_SOLITÁRIO= "FREQ_SOLITÁRIO"


FREQUENCIA_FALHA_MEMORIA= "FREQUENCIA_FALHA_MEMORIA"
TIPO_DIFICULDADE_COGNITIVA= "TIPO_DIFICULDADE_COGNITIVA"
QTD_DIFICULDADE_COGNITIVA= "QTD_DIFICULDADE_COGNITIVA"


HISTORICO_TRANSTORNO_ANSIEDADE= "HISTORICO_TRANSTORNO_ANSIEDADE"
MED_ANSIEDADE= "MED_ANSIEDADE"
FREQ_ANSIEDADE= "FREQ_ANSIEDADE"
INTENSIDADE_ANSIEDADE= "INTENSIDADE_ANSIEDADE"

FREQ_ANSIEDADE_2SEM= "FREQ_ANSIEDADE_2SEM"
FREQ_PREOCUPACAO_DESCONTROLADA_2SEM= "FREQ_PREOCUPACAO_DESCONTROLADA_2SEM"
FREQ_PREOCUPACAO_DESNECESSARIA_2SEM= "FREQ_PREOCUPACAO_DESNECESSARIA_2SEM"
FREQ_DIFICULDADE_DESCANSAR_2SEM= "FREQ_DIFICULDADE_DESCANSAR_2SEM"
FREQ_AGITACAO_INQUIETA= "FREQ_AGITACAO_INQUIETA"

mapa_limpo.update(
    {
    #Ansiedade, Depressão e seus Sintomas ########################################################

    "DEPEV_A": HISTORICO_DEPRESSAO,
    "DEPMED_A": REMEDIO_DEPRESSÃO, #pode enviesar muito!
    "DEPFREQ_A": FREQ_DEPRESSAO,
    "DEPLEVEL_A": INTENSIDADE_DEPRESSAO,
    "PHQ81_A": FREQ_PERDA_INTERESSE_2SEM,
    "PHQ82_A": FREQ_SENTIU_DEPRIMIDO_2SEM,
    #"HOPELESS_A": "k6_freq_desesperanca", #infelizmente não tem 2025. PHQ82_A equivalente?
    #"WORTHLESS_A": "k6_freq_desvalorizacao", #infelizmente não tem 2025. PHQ82_A equivalente?
    "GAD72_A": FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM,
    "PHQ83_A": FREQ_DIFICULDADE_DORMIR_2SEM,
    #"SLPFLL_A": "dificuldade_adormecer", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPSTY_A": "dificuldade_manter_sono", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPREST_A": "acorda_descansado", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    "PHQ84_A": FREQ_CANSASO_2SEM,
    #"FGEFRQTRD_A": "frequencia_fadiga_3m", #infelizmente não tem no paper e 2025. PHQ84_A equivalente?
    "PHQ85_A": FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM,
    "PHQ86_A": FREQ_AUTOAVALIACAO_NEGTIVA_2SEM, #auto estima, etc...
    "PHQ87_A": FREQ_DIFICULDADE_CONCENTRACAO_2SEM,
    #"SAD_A": "k6_freq_tristeza_profunda", #infelizmente não tem 2025. PHQ82_A equivalente?
    "LONELY_A": FREQ_SOLITÁRIO, #não está no paper mas deveria
    #"EFFORT_A": "k6_freq_esforco_extremo", não está no paper. Não achei equivalente
    #"PHQCAT_A": "escala_sintomas_depressão", #pode enviesar muito!
    
    "COGFRQDFF_A": FREQUENCIA_FALHA_MEMORIA, 
    "COGTYPEDFF_A": TIPO_DIFICULDADE_COGNITIVA,
    "COGMEMDFF_A": QTD_DIFICULDADE_COGNITIVA,

    "ANXEV_A": HISTORICO_TRANSTORNO_ANSIEDADE,
    "ANXMED_A": MED_ANSIEDADE, #pode enviesar muito!
    "ANXFREQ_A": FREQ_ANSIEDADE,
    "ANXLEVEL_A": INTENSIDADE_ANSIEDADE,

    "GAD71_A": FREQ_ANSIEDADE_2SEM,
    #"NERVOUS_A": "k6_freq_nervosismo",
    "GAD72_A": FREQ_PREOCUPACAO_DESCONTROLADA_2SEM,
    "GAD73_A": FREQ_PREOCUPACAO_DESNECESSARIA_2SEM,
    "GAD74_A": FREQ_DIFICULDADE_DESCANSAR_2SEM,
    "GAD75_A": FREQ_AGITACAO_INQUIETA, # não tinha no paper mas é de ansiedade

    #"GADCAT_A": "escala_sintomas_ansiedade", #pode enviesar muito!
    #"K6SPD_A": "sofrimento_psicologico_k6", #não tem no paper nem 2025. Pode enviesar?
    ##############################################################################################################
    
    # Segurança alimentar: INFELIZMENTE NÃO TEM EM 2025
    #"FDSLESS_A": "comeu_menos_que_devia", 
    #"FDSWEIGHT_A": "perdeu_peso_restricao_alimentar", 
    #"FDSSKIP_A": "diminuiu_ou_pulou_refeicoes",
    #"FDSHUNGRY_A": "passou_fome_por_falta_dinheiro",
    
    }
)


check = [mapa_limpo == MAPA_VARIAVEIS_PAPER]

check


NameError: name 'mapa_limpo' is not defined

In [ ]:
#TESTES APRENDIZADO ONE-HOT ENCODE

df_limpo[REGIAO_GEOGRAFICA] = df_limpo[REGIAO_GEOGRAFICA].replace({
    1: 'Northeast',
    2: 'Midwest',
    3: 'South',
    4: 'West'
})

df_limpo = pd.get_dummies(df_limpo, columns=[REGIAO_GEOGRAFICA], drop_first=True, dtype=int)

df_limpo

,categoria_razao_pobreza,qtd_anos_estados_unidos,estado_civil,cojuge_trabalha,sexo_conjuge,categoria_imc,peso_libras,visitas_pronto_atendimento_12m,visitas_emergencia_hospitalar_12m,escolaridade_maxima_familia,...,historico_hipertensao,satisfacao_com_a_vida,autoavaliacao_saude_geral,peso_amostral,id_domicilio,razao_renda_pobreza,target_medicacao,regiao_geografica_Northeast,regiao_geografica_South,regiao_geografica_West
0,6,NaN,1,2.0,1.0,3,171,0,0,4,...,2,2,3,10636.862,H012128,1.58,1.0,0,1,0
1,12,NaN,5,NaN,NaN,2,140,0,0,3,...,1,1,4,2996.860,H000617,4.32,0.0,0,1,0
2,1,NaN,7,NaN,NaN,3,175,0,0,2,...,2,2,3,29032.445,H033243,0.18,0.0,0,1,0
3,14,NaN,1,2.0,2.0,3,215,0,0,5,...,1,1,4,7875.107,H049140,6.32,0.0,0,1,0
4,13,NaN,7,NaN,NaN,4,198,0,0,8,...,1,2,2,4824.620,H012797,4.90,0.0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24210,14,NaN,1,1.0,1.0,4,200,0,0,9,...,1,1,3,4869.749,H014073,6.18,0.0,0,0,1
24211,4,NaN,5,NaN,NaN,4,285,0,2,5,...,2,2,2,3350.243,H029805,1.21,0.0,0,0,1
24212,12,NaN,4,NaN,NaN,4,210,0,2,4,...,1,1,5,2374.594,H003491,4.32,0.0,0,0,1
24213,9,NaN,1,2.0,1.0,3,165,0,0,5,...,2,1,2,2400.569,H001251,2.63,0.0,0,0,1
